# BUSI2066 · Week 8 — Two Numbers at Once: Correlation


### Before you start

Drag in **this notebook** and **`customers.csv`**, and select the kernel.

### Today

Until now the course was about **one column**, or one column split into groups. From today
you ask about **two columns at once** — and this week both of them are numbers.

> **Do customers who live further from the store spend less in it?**

That is a **relational** research question (week 4). Today you learn to draw it, measure it, test it,
and — the part people skip — say what it does *not* tell you.

### The rule

> **AI explains. It does not write** — except where marked ⭐.

---
# Part 0 · Set up, and a word about this file

In [ ]:
%pip install -q pandas seaborn scipy statsmodels

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

df = pd.read_csv('customers.csv')

print(df.shape)
df.head()

**1,000 customers of a retailer that sells both online and in a physical store.**

| Column | What it is |
|---|---|
| `age` · `credit.score` | Two things the retailer knows about each customer |
| `email` | Whether they gave an email address — `yes` / `no` |
| `distance.to.store` | How far they live from the store, in km |
| `online.visits` · `online.trans` · `online.spend` | Visits, transactions and spend on the website |
| `store.trans` · `store.spend` | Transactions and spend in the physical store |
| `sat.service` · `sat.selection` | Two satisfaction ratings, 1–5 |

⚠️ **This file is simulated.** Somebody wrote a program that generates plausible customers, and the
patterns you will find were put there on purpose. That is why it is clean enough to teach with. Your
survey data will be messier — but the method is identical.

In [ ]:
df.info()

`sat.service` and `sat.selection` have **659 values out of 1,000** — 341 customers never rated. Keep
that in mind; it matters in Part 3.

---
# Part 1 · Draw it first

Three research questions about this retailer, in the form week 4 taught you:

| RQ | Question | The retailer's hypothesis |
|---|---|---|
| **RQ1** | Is age related to credit score? | **H1** · Older customers tend to have a higher credit score |
| **RQ2** | Is online spend related to in-store spend? | **H2** · Customers who buy more online tend to buy **less** in store |
| **RQ3** | Is distance to the store related to in-store spend? | **H3** · The closer customers live, the **more** they spend in store |

All three are **relational** questions (week 4). Each hypothesis also names a **direction** — keep
that in mind for Part 3.

**Never test a relationship you have not looked at.** The chart for two numbers is the **scatter
plot** — one dot per customer, one column on each axis.

## 1.1 RQ1 — age and credit score

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='age', y='credit.score')
plt.title('RQ1 — Credit score against age')
plt.show()

A cloud that leans **upward and to the right**, loosely. Older customers tend to have higher credit
scores, but with a lot of scatter — plenty of young customers with high scores and vice versa.

That is what a **weak positive** relationship looks like. Hold the picture; you will put a number on
it in Part 2.

## 1.2 RQ2 — online spend and store spend

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='store.spend', y='online.spend')
plt.title('RQ2 — Online spend against in-store spend')
plt.show()

This one is hard to read. A few huge spenders stretch both axes, crowding most smaller values into
the corner. Both spending columns are strongly right-skewed.

**Why try log here?** It compresses large values so the crowded points become easier to distinguish.
We then check the scatter plot: does the relationship look closer to a straight line? Less crowding
does not guarantee a relationship. `np.log` is numpy's natural log.

In [ ]:
print(df['online.spend'].min(), df['store.spend'].min())

Both minimums are **0**: 368 customers spent nothing online and 381 spent nothing in store.
`log(0)` is undefined. **For this exercise**, we use `log(x + 1)` so zero spending becomes
`log(1) = 0` and those customers stay in the analysis.

Add 1 to every value in the column, not just the zeros. Adding 1 is a modelling choice, not a
universal rule. Its effect depends on the units and matters most for values near zero.

In [ ]:
df['log_online'] = np.log(df['online.spend'] + 1)
df['log_store']  = np.log(df['store.spend'] + 1)

plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='log_store', y='log_online')
plt.xlabel('log(store spend + 1)')
plt.ylabel('log(online spend + 1)')
plt.title('RQ2 — after log transformation')
plt.show()

### What the log did to one column

Look at `store.spend` on its own, before and after.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['store.spend'], bins=40, ax=axes[0])
axes[0].set_title('store.spend — raw')

sns.histplot(df['log_store'], bins=40, ax=axes[1])
axes[1].set_title('log(store.spend + 1)')

plt.show()

📖 `plt.subplots(1, 2)` draws two charts side by side — one row, two columns — and `ax=axes[0]` says
which one to draw into. You saw it in week 6 as read-level code. Copy it when you want two charts
together.

**Left:** a long tail to the right — right-skewed. **Right:** the customers who spent something now
sit in a rough hump. The tall bar at 0 is the 381 who spent nothing (log(0 + 1) = 0).

**What next? Check the scatter plot.** A more symmetric histogram does not guarantee a straight
relationship between two columns.

| What you see in the scatter plot | What to do |
|---|---|
| A roughly straight pattern | The transformation may be suitable. Interpret the transformed variables. |
| A clear curved pattern | Consider another transformation that suits that curve, then plot again. |
| A shapeless cloud | There may be little relationship. Do not keep trying just to get significance. |

Square root, square, inverse, exp and sigmoid change shapes in different ways. They are not
interchangeable fixes. For example, if the pattern is approximately `y = a + b * x²`, plotting y
against x² may straighten it. If it is approximately `y = a + b / x`, plotting y against 1/x may help.
These are pattern examples, not instructions to try every transformation.

Explain any choice made after exploring the data. We focus on log here because the raw spending
plots are crowded and log makes them easier to inspect.

[Guidance: NIST, transformations to improve fit](https://www.itl.nist.gov/div898/handbook/pmd/section6/pmd624.htm)


Now the cloud is easier to inspect. There are customers who spent nothing in store, customers who
spent nothing online, and otherwise a shapeless cloud with no clear downward pattern. The data do
not support the hypothesis that customers who buy more online buy less in store.

**Should we try another transformation?** Only if a clear pattern gives us a reason. A remaining
curve may suggest another transformation. A shapeless cloud may mean there is little relationship.
Here, log helped us see the cloud; finding no relationship does not mean it failed.

## 1.3 RQ3 — distance and store spend

In [ ]:
df['log_dist'] = np.log(df['distance.to.store'])     # no zeros here, so no +1 needed

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.scatterplot(data=df, x='distance.to.store', y='store.spend', ax=axes[0])
axes[0].set_title('RQ3 — raw')

sns.scatterplot(data=df, x='log_dist', y='log_store', ax=axes[1])
axes[1].set_title('RQ3 — both logged')

plt.show()

📖 `plt.subplots` again, as in §1.2.

Left: unreadable, for the same reason as RQ2. Right: a clear **downward** slope. Customers further
away spend less in store — and the log scale shows it where the raw scale hid it.

**The transformation did not create the relationship.** It was always there. It made it visible.

---
# Part 2 · Put a number on it

## 2.1 Covariance

**By hand first, on five points.** Covariance asks: when x is above its mean, is y above its mean too?

$$\text{cov}(x, y) = \frac{\sum (x - \bar{x})(y - \bar{y})}{n - 1}$$

| x | y | x − x̄ | y − ȳ | product |
|---|---|---|---|---|
| 2 | 10 | −1.0 | −4.2 | 4.20 |
| 3 | 14 | 0.0 | −0.2 | 0.00 |
| 2.7 | 12 | −0.3 | −2.2 | 0.66 |
| 3.2 | 15 | 0.2 | 0.8 | 0.16 |
| 4.1 | 20 | 1.1 | 5.8 | 6.38 |
| | | | **sum** | **11.40** |

The means are x̄ = 3 and ȳ = 14.2. So cov = 11.40 ÷ 4 = **2.85**. Positive: when x is high, y tends to
be high too. Same side of the mean → positive product; opposite sides → negative. **n − 1**, as for
the standard deviation in week 5.

Now let pandas do it on the real data — RQ1, RQ2, RQ3:

In [ ]:
print(df['age'].cov(df['credit.score']))                # RQ1
print(df['online.spend'].cov(df['store.spend']))        # RQ2
print(df['distance.to.store'].cov(df['store.spend']))   # RQ3

**63.2**, **−1,151.6** and **−376.5**. Positive means they rise together, negative means one rises as
the other falls.

But the *size* is meaningless on its own. 63 is in "years × credit points"; −376 is in "km × dollars".
You cannot compare them, and you cannot say whether 63 is big. RQ2's −1,152 looks the biggest — §2.2
shows it is the weakest. **Covariance tells you the direction and nothing else.**

Every pair at once is a **covariance matrix**:

In [ ]:
numbers = df.drop(columns=['cust.id', 'email', 'log_online', 'log_store', 'log_dist'])

numbers.cov().round(1)

`drop(columns=[...])` removes columns — `cust.id` because an ID is not a quantity, `email` because it
is text, and our three logged columns because they would duplicate their originals.

The diagonal is each column's **variance** (the standard deviation squared). Every other cell is a
covariance, in its own units. Hard to compare — so we scale it.

## 2.2 Correlation — the number you will actually use

In [ ]:
print(df['age'].corr(df['credit.score']))                # RQ1
print(df['online.spend'].corr(df['store.spend']))        # RQ2
print(df['distance.to.store'].corr(df['store.spend']))   # RQ3

| RQ | r | Reading |
|---|---|---|
| 1 | **0.25** | weak positive |
| 2 | **−0.05** | essentially nothing |
| 3 | **−0.24** | weak negative |

The **Pearson correlation coefficient, r**, is covariance divided by both standard deviations:

$$r = \frac{\text{cov}(x, y)}{s_x \, s_y}$$

On the five points: 2.85 ÷ (0.76 × 3.77) = **0.99** — nearly a perfect line. That scaling forces r
into **−1 to +1**, with no units — so 0.25 means the same thing whatever you are measuring. Check it
on RQ1 in the next cell.

| r | Means |
|---|---|
| **+1** | perfect straight line, upward |
| **0** | no *linear* relationship |
| **−1** | perfect straight line, downward |

Rough guide for strength uses **|r|**, the value without its sign: from 0 to below 0.3 is weak,
0.3 to 0.7 is moderate, above 0.7 is strong. For example, r = -1 is strong and negative.
Business data lives mostly in the weak zone, and a weak correlation across a thousand customers is
often worth money.

⚠️ **The word *linear* is doing work.** r measures how well a straight line fits. Two columns can be
tightly related along a curve and have r ≈ 0. Which is why Part 1 came first.

In [ ]:
df['age'].cov(df['credit.score']) / (df['age'].std() * df['credit.score'].std())

**0.2545** — the same number `.corr()` gave. That is all r is.

## 2.3 All of them at once

In [ ]:
numbers.corr().round(2)

`.corr()` on the whole table gives a **correlation matrix**: every column against every other. The
diagonal is 1 (a column against itself). It is symmetric, so read one triangle.

Too many numbers to read? Draw it.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(numbers.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix')
plt.show()

Red is positive, blue is negative, white is nothing. `center=0` makes white mean zero; `annot=True`
prints the numbers. Three things jump out that a table hides:

- **`online.visits`, `online.trans` and `online.spend` are correlated at 0.98–0.99.** They are three
  measurements of one behaviour. In a regression (week 11) you would use *one* of them.
- **`store.trans` and `store.spend` at 0.89** — same story.
- **Almost everything else is white.** Age is unrelated to spending. Distance is unrelated to online
  behaviour. A heatmap tells you what *not* to bother testing.

## 2.4 Every pair, drawn

In [ ]:
sns.pairplot(df[['age', 'credit.score', 'distance.to.store', 'store.spend', 'online.spend', 'email']])
plt.show()

`pairplot` draws a scatter plot for every pair and a histogram on the diagonal. We gave it six
columns, but `email` is text, so `pairplot` skips it: you get a 5 × 5 grid. It is the fastest way to
*look* at a dataset you have never seen — use it on your survey export before you decide anything.
More than about six columns and the dots get too small to read.

---
# Part 3 · Is it real? — the correlation test

r = 0.25 describes **this sample**. The question the retailer cares about is whether age and credit
score are related in the **population** — and week 6 taught you the shape of that question.

| | |
|---|---|
| **H₀** | ρ = 0 — no linear relationship in the population |
| **H₁** | ρ ≠ 0 — there is one |

(ρ, "rho", is the population correlation. r is the sample's.)

**Where p comes from.** The test turns r into **t = r ÷ SE(r)** — how many standard errors r is from
0. A curve called the **t-distribution** (Gosset, 1908 — see the slides) says how often a t that far
out turns up *if H₀ is true*. That is p. `pearsonr` does all of it for you.

In [ ]:
result = pearsonr(df['age'], df['credit.score'])

print("r:", result.statistic)
print("p:", result.pvalue)
print("95% CI:", result.confidence_interval(0.95))

| | Value | Reading |
|---|---|---|
| r | **0.254** | weak positive, as the picture said |
| p | **3 × 10⁻¹⁶** | far below 0.05 — reject H₀ |
| 95% CI | **0.196 → 0.312** | a range consistent with the data; it excludes zero |

**Conclusion:** the test gives evidence of a nonzero population correlation. The interval describes its size and uncertainty; it is near the weak to moderate boundary.

Same test, RQ2:

In [ ]:
result = pearsonr(df['online.spend'], df['store.spend'])

print("r:", result.statistic)
print("p:", result.pvalue)
print("95% CI:", result.confidence_interval(0.95))

RQ2: r = −0.052, **p = 0.102**, 95% CI **−0.113 to 0.010**.

- **Test:** fail to reject H₀: ρ = 0. We lack sufficient evidence of a nonzero population correlation.
  This does not prove zero and does not make the calculation untrustworthy.
- **Interval:** all values in the interval are weak correlations. Strong linear correlations lie
  outside it. This information comes from the interval, not merely from p being above 0.05.

A wide interval could include zero and strong correlations. Here the narrow interval is informative.
Report both results. Do not describe this as proof of no relationship, or keep changing the analysis
until it becomes significant. 95% is the confidence level; the endpoints describe correlation size.

### The retailer had a direction in mind

H2 said *more online, **less** in store* — that is H₁: ρ < 0, a **one-sided** test. Week 6 showed the
argument: `alternative='less'` or `'greater'`. 📖 Read-level.

In [ ]:
result = pearsonr(df['online.spend'], df['store.spend'], alternative='less')
print("one-sided p:", result.pvalue)

**p = 0.051** — half of the two-sided 0.10, because r pointed the way H2 guessed. Still above 0.05, so we
**still fail to reject H₀**.

That halving is why the direction must be written down **before** you look at the data. Choose it
after, and you have quietly doubled your chance of a false "yes". **In this course, use two-sided**
unless your plan named the direction first.

## 3.1 The test on transformed columns

RQ2 on the logged columns:

In [ ]:
logged2 = pearsonr(df['log_store'], df['log_online'])
print("RQ2 logged  r =", round(logged2.statistic, 3), "  p =", round(logged2.pvalue, 3))

**r = 0.016, p = 0.603**, 95% CI **−0.046 to 0.078**, on the logged columns.
The test again fails to reject zero; the interval again contains only weak correlations. A larger
p-value does not mean the null hypothesis is more true. Log helped us inspect the plot, but no clear
curve remains to justify another transformation. Do not search for significance.

RQ3 on the raw columns, then on the logged ones:

In [ ]:
raw = pearsonr(df['distance.to.store'], df['store.spend'])
logged = pearsonr(df['log_dist'], df['log_store'])

print("raw     r =", round(raw.statistic, 3), "  p =", raw.pvalue)
print("logged  r =", round(logged.statistic, 3), "  p =", logged.pvalue)

**−0.24 raw, −0.45 logged.** Both significant — but the logged version is nearly twice as strong,
because the straight line fits the logged cloud far better than the raw one (look back at §1.3).

Two rules for transformations:

1. **Choose a transformation for a visible pattern**, then check whether the relationship is
   closer to a straight line. Pearson's r measures linear association. A larger r or smaller p alone
   does not establish that the choice is appropriate.
2. **Then interpret in the transformed units, honestly.** −0.45 is the correlation between *log*
   distance and *log* spend. Saying "distance and spend correlate at −0.45" is a small lie.

## 3.2 When the test refuses to run

In [ ]:
result = pearsonr(df['sat.service'], df['sat.selection'])
print(result.statistic)

`nan` — not a number. `pearsonr` cannot cope with missing values, and 341 customers never rated.
`.corr()` quietly skipped them; the test does not.

This is the week-4 decision again, and `dropna()` is the answer. But **drop only the rows you need
to** — `dropna()` on the whole table would also remove anyone missing anything else.

In [ ]:
rated = df.dropna(subset=['sat.service', 'sat.selection'])
print(len(rated))

result = pearsonr(rated['sat.service'], rated['sat.selection'])
print("r:", round(result.statistic, 3), "  p:", result.pvalue)

`subset=[...]` restricts `dropna` to those two columns: **659 customers remain**, and the two
satisfaction ratings correlate at **0.59** — moderate and positive, which is what you would hope for
two questions about the same shopping experience.

⚠️ **Your survey export will do this to you.** Any respondent who skipped one question breaks
`pearsonr` for every pair that question is in. `dropna(subset=...)` per test is the fix.

---
# Part 4 · What correlation does not tell you

Three sentences, and your report will be marked on all three.

**1 — Association is not causation.** Week 4, again. Age correlates with credit score. Does getting
older *raise* your credit score? Perhaps — or older people have simply had longer to build one, or
the retailer's older customers are a different kind of person. The number cannot distinguish these.

**2 — r says nothing about the slope.** r = 0.25 could mean credit score rises 1 point per year of age
or 50 points — it measures how tightly the dots hug a line, not how steep the line is. The slope is
week 11's job.

**3 — A significant p is not a big r.** With 1,000 customers, any r beyond ±0.062 is significant at
5% — r = 0.07 gives p = 0.03. An r that small is rarely worth acting on. Report both, and let the
reader see the size.

---
# 🛠 Exercise · Two more relationships

Eight questions. **Complete all eight questions in class.** No answers are printed here: run the code and read your own output.

**1 · Draw it.** Scatter plot of `online.visits` (x) against `online.spend` (y), with labels and a title.
Describe the shape in one phrase.

_(your answer)_

In [ ]:
# Your code here

**2 · Measure it.** Correlation between `online.visits` and `online.spend`.

Now say, in one sentence, why a correlation this high is **not** an interesting finding.

_(your answer)_

In [ ]:
# Your code here

**3 · Test it.** Does `age` relate to `store.spend`? State H₀ and H₁, run the test, report r, p and
the interval, and conclude.

- **H₀:** _(your answer)_
- **H₁:** _(your answer)_
- **Conclusion:** _(your answer)_

In [ ]:
# Your code here

**4 · Transform and re-test.** `online.trans` is a count and is right-skewed. Add a column
`log_trans = log(online.trans + 1)`, then test `log_trans` against `log_online`.

Why the `+ 1`? _(your answer)_

In [ ]:
# Your code here

**5 · The missing rows.** Test whether `sat.service` relates to `store.spend`.

How many rows did you have to drop, and how did you make sure you dropped only those?

_(your answer)_

In [ ]:
# Your code here

**6 · Read the heatmap.** From §2.3, name **one** pair of columns you would *not* bother testing, and
say why.

_(your answer)_

**7 · The sentence for the report.** RQ3 gave r = −0.45 between log distance and log store spend,
p < 0.001. Write the sentence that goes in the results section — **and then the sentence that goes
in the limitations section**.

- Results: _(your answer)_
- Limitations: _(your answer)_

**8 · Put it together.** The store manager says: *"Customers who live far away hardly ever shop in
the store. Should we stop posting them our printed flyer?"*

Use `store.trans` — how many times each customer bought something in the store. Do the four steps
in order. Each step uses what the step before it told you.

**a · Look.** Draw a scatter plot of `distance.to.store` (x) against `store.trans` (y), with labels
and a title. Can you see a pattern? Is either column skewed? Draw a histogram if you are not sure.

_(your answer)_

**b · Transform, if (a) says you should.** Make a logged column for `store.trans`. Check its smallest
value first: does it need `+ 1`? For distance, use `log_dist` from §1.3. Draw the scatter plot again.

_(your answer)_

**c · Test.** Run `pearsonr` on the version you chose in (b). Report r, p and the 95% interval.

- **H₀:** _(your answer)_
- **H₁:** _(your answer)_
- **Result:** _(your answer)_

**d · Decide.** Write two or three sentences for the manager. Is there a relationship, and how
strong is it? What can this file **not** tell the manager about the flyer? What do you advise?

_(your answer)_

In [ ]:
# Your code here

---
# ⭐ Finished early?

**Optional and unassessed.**

In §3.1 the log transformation strengthened RQ3 from −0.24 to −0.45. Try a third version: the correlation between **`1 / distance`** and raw `store.spend`.

Run it, and look at the sign of r.

Ask your AI why a *negative* relationship became *positive*, and what "1 over distance" actually
means as a quantity. Then decide: which of the three versions — raw, log, or inverse — would you put
in front of the retailer, and why?

_(your answer)_

In [ ]:
# Your code here

---
# ✅ Done — hand it in

**No cell is designed to fail.** Run All should go straight through — `pairplot` takes a few seconds.

1. **Run All**, no red.
2. Save, **commit and sync**.
3. On Moodle: **this notebook file**.
4. **Stop your Codespace**.

## What you can now do

- [ ] draw a scatter plot **before** testing anything
- [ ] work out a covariance by hand, and explain why it has a direction but no usable size
- [ ] read r: sign, strength, and the word *linear*
- [ ] build a correlation matrix and a heatmap, and use them to decide what not to test
- [ ] run `pearsonr`, state H₀ and H₁, and read r, p and the interval
- [ ] log-transform a skewed column, with `+ 1` when it has zeros, check it with a histogram, and interpret honestly
- [ ] know that `alternative='less'` / `'greater'` exists, and why the direction is chosen first
- [ ] drop only the rows a test needs, with `dropna(subset=...)`
- [ ] say what correlation does not tell you — causation, slope, and size

## Coming up

**Next week:** two *categories* at once — the chi-square test.